In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [3]:
def load_dataset(label_count):

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [4]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels = load_dataset(8)

45597
36477
9120
Load Data for 8-Label MultiLabel success!!


In [5]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
cfg_train_loader = DataLoader(data_list[:split_idx], batch_size=64, shuffle=False)
cfg_test_loader = DataLoader(data_list[split_idx:], batch_size=64, shuffle=False)


In [6]:

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SourceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
opcode_train_dataset = OpcodeDataset(X_train_opcode, y_train)
opcode_test_dataset = OpcodeDataset(X_test_opcode, y_test)

opcode_train_loader = DataLoader(opcode_train_dataset, batch_size=64, shuffle=False)
opcode_test_loader  = DataLoader(opcode_test_dataset, batch_size=64, shuffle=False)

source_train_dataset = SourceDataset(X_train_source, y_train)
source_test_dataset  = SourceDataset(X_test_source, y_test)

source_train_loader = DataLoader(source_train_dataset, batch_size=64, shuffle=False)
source_test_loader  = DataLoader(source_test_dataset, batch_size=64, shuffle=False)

In [8]:
import torch
import torch.nn as nn

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=34000,
        embedding_dim=256,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        return x
        x = self.out(x)
        x = self.sigmoid(x)
        
        return x
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class GAT(nn.Module):
    def __init__(
        self,
        in_channels=57,
        hidden_channels=64,
        num_classes=8
    ):
        super(GAT, self).__init__()

        # ===== GAT Layers =====
        self.conv1 = GATConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=4,
            dropout=0.3
        )

        self.conv2 = GATConv(
            in_channels=hidden_channels * 4,
            out_channels=hidden_channels,
            heads=2,
            dropout=0.3
        )

        # ===== Batch Normalization =====
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)

        # ===== Dense Layers =====
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)

        # ===== Output Layer =====
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):

        # ===== Inputs =====
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # ===== GAT Layer 1 =====
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== GAT Layer 2 =====
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Global Pooling =====
        x = global_mean_pool(x, batch)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Dense Layer =====
        x = self.fc1(x)
        x = F.relu(x)
        return x

        # ===== Output =====
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

In [9]:
codebert_model = load_model('DATASET/bert.h5')
codebert_model = Model(
    inputs=codebert_model.input,
    outputs=codebert_model.layers[-2].output   # layer 64
)

bilstm_model = BiLSTMModel()
bilstm_model.load_state_dict(torch.load("DATASET/bilstm_model.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"
bilstm_model.to(device)
bilstm_model.eval()

gat_model = GAT()
gat_model.load_state_dict(torch.load("DATASET/gat.pth"))
gat_model.to(device)
gat_model.eval()

C:\Users\Admin\AppData\Local\Temp\ipykernel_16476\1889956571.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bilstm_model.load_state_dict(torch.load("DATASET/bilstm_mode

GAT(
  (conv1): GATConv(57, 64, heads=4)
  (conv2): GATConv(256, 64, heads=2)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=8, bias=True)
)

In [10]:
for i, layer in enumerate(codebert_model.layers):
    try:
        print(i, layer.name, layer.output_shape)
    except:
        print(i, layer.name, "no shape")

0 sourcecode_features no shape
1 batch_normalization_3 no shape
2 dense_6 no shape
3 dropout_4 no shape
4 dense_7 no shape
5 batch_normalization_4 no shape
6 dropout_5 no shape
7 dense_8 no shape
8 dropout_6 no shape
9 dense_9 no shape
10 batch_normalization_5 no shape
11 dropout_7 no shape
12 dense_10 no shape


In [11]:
class MetaClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 8),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [12]:
def extract_opcode_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)  # (N, 8)


def extract_gat_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)


codebert_features_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_features_test  = codebert_model.predict(X_test_source, batch_size=64)

opcode_feat_train = extract_opcode_features(bilstm_model, opcode_train_loader, device)
opcode_feat_test  = extract_opcode_features(bilstm_model, opcode_test_loader, device)

gat_feat_train = extract_gat_features(gat_model, cfg_train_loader, device)
gat_feat_test  = extract_gat_features(gat_model, cfg_test_loader, device)

# CodeBERT (Keras)
codebert_feat_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_feat_test  = codebert_model.predict(X_test_source, batch_size=64)

# convert numpy -> torch nếu cần
codebert_feat_train = torch.tensor(codebert_feat_train)
codebert_feat_test  = torch.tensor(codebert_feat_test)

train_features = torch.cat([
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train
], dim=1)   # (N, 24)

test_features = torch.cat([
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test
], dim=1)

570/570 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
570/570 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [13]:
import torch
import torch.nn as nn

class TrueHierarchicalFusion(nn.Module):
    def __init__(self, seq_len=8, feature_dim=8, num_heads=4, num_classes=8):
        super().__init__()
        
        self.seq_len = seq_len
        self.feature_dim = feature_dim
        
        # ===== LEVEL 1: Self-Attention for each modality =====
        # Opcode self-attention
        self.opcode_self_attn = nn.MultiheadAttention(
            embed_dim=feature_dim, 
            num_heads=num_heads, 
            batch_first=True,
            dropout=0.1
        )
        self.opcode_norm = nn.LayerNorm(feature_dim)
        
        # GAT self-attention
        self.gat_self_attn = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )
        self.gat_norm = nn.LayerNorm(feature_dim)
        
        # CodeBERT self-attention
        self.codebert_self_attn = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )
        self.codebert_norm = nn.LayerNorm(feature_dim)
        
        # ===== LEVEL 2: Pairwise Cross-Attention Fusion =====
        # Fusion 1: Opcode + GAT
        self.fusion_og_query = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_og_key = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_og_norm1 = nn.LayerNorm(feature_dim)
        self.fusion_og_norm2 = nn.LayerNorm(feature_dim)
        self.fusion_og_proj = nn.Sequential(
            nn.Linear(seq_len * feature_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Fusion 2: GAT + CodeBERT
        self.fusion_gc_query = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_gc_key = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_gc_norm1 = nn.LayerNorm(feature_dim)
        self.fusion_gc_norm2 = nn.LayerNorm(feature_dim)
        self.fusion_gc_proj = nn.Sequential(
            nn.Linear(seq_len * feature_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Fusion 3: Opcode + CodeBERT
        self.fusion_oc_query = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_oc_key = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads, batch_first=True, dropout=0.1
        )
        self.fusion_oc_norm1 = nn.LayerNorm(feature_dim)
        self.fusion_oc_norm2 = nn.LayerNorm(feature_dim)
        self.fusion_oc_proj = nn.Sequential(
            nn.Linear(seq_len * feature_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # ===== LEVEL 3: Final High-level Fusion =====
        self.final_fusion_attn = nn.MultiheadAttention(
            embed_dim=64, num_heads=4, batch_first=True, dropout=0.1
        )
        self.final_fusion_norm = nn.LayerNorm(64)
        self.final_fusion_proj = nn.Sequential(
            nn.Linear(64 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # ===== Final Classifier =====
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
            nn.Sigmoid()
        )

    def forward(self, opcode, gat, codebert):
        # Reshape from (B, 64) to (B, 8, 8)
        opcode = opcode.view(-1, self.seq_len, self.feature_dim)      # (B, 8, 8)
        gat = gat.view(-1, self.seq_len, self.feature_dim)            # (B, 8, 8)
        codebert = codebert.view(-1, self.seq_len, self.feature_dim)  # (B, 8, 8)
        
        # ===== LEVEL 1: Self-Attention =====
        opcode_attn, _ = self.opcode_self_attn(opcode, opcode, opcode)
        opcode = self.opcode_norm(opcode + opcode_attn)
        
        gat_attn, _ = self.gat_self_attn(gat, gat, gat)
        gat = self.gat_norm(gat + gat_attn)
        
        codebert_attn, _ = self.codebert_self_attn(codebert, codebert, codebert)
        codebert = self.codebert_norm(codebert + codebert_attn)
        
        # ===== LEVEL 2: Pairwise Fusion =====
        # Fusion 1: Opcode + GAT
        fusion_og_q, _ = self.fusion_og_query(opcode, gat, gat)
        fusion_og_q = self.fusion_og_norm1(opcode + fusion_og_q)
        fusion_og_k, _ = self.fusion_og_key(gat, opcode, opcode)
        fusion_og_k = self.fusion_og_norm2(gat + fusion_og_k)
        fusion_og_cat = torch.cat([fusion_og_q, fusion_og_k], dim=1)
        fusion_og_flat = fusion_og_cat.view(fusion_og_cat.size(0), -1)
        fusion_og_out = self.fusion_og_proj(fusion_og_flat)  # (B, 64)
        
        # Fusion 2: GAT + CodeBERT
        fusion_gc_q, _ = self.fusion_gc_query(gat, codebert, codebert)
        fusion_gc_q = self.fusion_gc_norm1(gat + fusion_gc_q)
        fusion_gc_k, _ = self.fusion_gc_key(codebert, gat, gat)
        fusion_gc_k = self.fusion_gc_norm2(codebert + fusion_gc_k)
        fusion_gc_cat = torch.cat([fusion_gc_q, fusion_gc_k], dim=1)
        fusion_gc_flat = fusion_gc_cat.view(fusion_gc_cat.size(0), -1)
        fusion_gc_out = self.fusion_gc_proj(fusion_gc_flat)  # (B, 64)
        
        # Fusion 3: Opcode + CodeBERT
        fusion_oc_q, _ = self.fusion_oc_query(opcode, codebert, codebert)
        fusion_oc_q = self.fusion_oc_norm1(opcode + fusion_oc_q)
        fusion_oc_k, _ = self.fusion_oc_key(codebert, opcode, opcode)
        fusion_oc_k = self.fusion_oc_norm2(codebert + fusion_oc_k)
        fusion_oc_cat = torch.cat([fusion_oc_q, fusion_oc_k], dim=1)
        fusion_oc_flat = fusion_oc_cat.view(fusion_oc_cat.size(0), -1)
        fusion_oc_out = self.fusion_oc_proj(fusion_oc_flat)  # (B, 64)
        
        # ===== LEVEL 3: High-level Fusion =====
        # Reshape outputs to (B, 1, 64) for attention
        fusion_og_out_3d = fusion_og_out.unsqueeze(1)  # (B, 1, 64)
        fusion_gc_out_3d = fusion_gc_out.unsqueeze(1)  # (B, 1, 64)
        fusion_oc_out_3d = fusion_oc_out.unsqueeze(1)  # (B, 1, 64)
        
        # Concatenate for multi-head attention
        all_fusions = torch.cat([fusion_og_out_3d, fusion_gc_out_3d, fusion_oc_out_3d], dim=1)  # (B, 3, 64)
        
        # Multi-head attention to fuse all 3 modality pairs
        fused_attn, _ = self.final_fusion_attn(all_fusions, all_fusions, all_fusions)
        fused = self.final_fusion_norm(all_fusions + fused_attn)
        
        # Flatten and project
        fused_flat = fused.view(fused.size(0), -1)  # (B, 192)
        final_features = self.final_fusion_proj(fused_flat)  # (B, 128)
        
        # ===== Final Classification =====
        outputs = self.classifier(final_features)
        
        return outputs

# Rename để compatible
HierarchicalFusion = TrueHierarchicalFusion

In [14]:
class FusionDataset(Dataset):
    def __init__(self, opcode, gat, codebert, y):
        self.opcode = opcode.float()
        self.gat = gat.float()
        self.codebert = codebert.float()
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.opcode[idx],
            self.gat[idx],
            self.codebert[idx],
            self.y[idx]
        )

In [15]:
train_dataset = FusionDataset(
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train,
    y_train
)

test_dataset = FusionDataset(
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test,
    y_test
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [16]:
def train_model(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for opcode, gat, codebert, y in train_loader:
        opcode = opcode.to(device)
        gat = gat.to(device)
        codebert = codebert.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(opcode, gat, codebert)

        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [17]:
from sklearn.metrics import classification_report
import numpy as np

def evaluate_model(model, test_loader, device):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for opcode, gat, codebert, y in test_loader:

            opcode = opcode.to(device)
            gat = gat.to(device)
            codebert = codebert.to(device)

            outputs = model(opcode, gat, codebert)

            all_preds.append(outputs.cpu())
            all_targets.append(y)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    preds_bin = (all_preds > 0.5).astype(int)

    print("\n===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====")
    print(classification_report(all_targets, preds_bin, digits=4))

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HierarchicalFusion().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 100

for epoch in range(EPOCHS):
    loss = train_model(model, train_loader, optimizer, criterion, device)

    if (epoch + 1) % 4 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss:.4f}")
    if (epoch + 1) % 10 == 0:
        evaluate_model(model, test_loader, device)

Epoch 4/100 - Loss: 0.0730
Epoch 8/100 - Loss: 0.0587

===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0     0.9406    0.9289    0.9347      6091
           1     0.7305    0.5126    0.6025       476
           2     0.9500    0.9679    0.9589      5755
           3     0.9070    0.8841    0.8954      1268
           4     0.7944    0.7250    0.7581      1018
           5     0.9399    0.8649    0.9009      3798
           6     0.8135    0.7904    0.8018       458
           7     0.9568    0.9381    0.9474      4039

   micro avg     0.9322    0.9067    0.9193     22903
   macro avg     0.8791    0.8265    0.8499     22903
weighted avg     0.9305    0.9067    0.9178     22903
 samples avg     0.8773    0.8610    0.8590     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 12/100 - Loss: 0.0528
Epoch 16/100 - Loss: 0.0488
Epoch 20/100 - Loss: 0.0449

===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0     0.9333    0.9345    0.9339      6091
           1     0.7339    0.5504    0.6291       476
           2     0.9479    0.9712    0.9594      5755
           3     0.9059    0.8888    0.8973      1268
           4     0.7926    0.7318    0.7610      1018
           5     0.9285    0.8781    0.9026      3798
           6     0.7794    0.7948    0.7870       458
           7     0.9496    0.9470    0.9483      4039

   micro avg     0.9256    0.9142    0.9199     22903
   macro avg     0.8714    0.8371    0.8523     22903
weighted avg     0.9240    0.9142    0.9187     22903
 samples avg     0.8760    0.8686    0.8622     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 24/100 - Loss: 0.0428
Epoch 28/100 - Loss: 0.0407

===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0     0.9395    0.9333    0.9364      6091
           1     0.7318    0.5504    0.6283       476
           2     0.9483    0.9717    0.9598      5755
           3     0.9124    0.8872    0.8996      1268
           4     0.7878    0.7515    0.7692      1018
           5     0.9387    0.8676    0.9018      3798
           6     0.7828    0.7948    0.7887       458
           7     0.9561    0.9436    0.9498      4039

   micro avg     0.9300    0.9125    0.9212     22903
   macro avg     0.8747    0.8375    0.8542     22903
weighted avg     0.9288    0.9125    0.9201     22903
 samples avg     0.8785    0.8669    0.8628     22903



...

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 84/100 - Loss: 0.0275
Epoch 88/100 - Loss: 0.0276

===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0     0.9305    0.9379    0.9342      6091
           1     0.7287    0.5756    0.6432       476
           2     0.9491    0.9717    0.9602      5755
           3     0.9173    0.9006    0.9089      1268
           4     0.7976    0.7200    0.7568      1018
           5     0.9387    0.8676    0.9018      3798
           6     0.7752    0.7904    0.7827       458
           7     0.9499    0.9490    0.9495      4039

   micro avg     0.9274    0.9145    0.9209     22903
   macro avg     0.8734    0.8391    0.8547     22903
weighted avg     0.9260    0.9145    0.9197     22903
 samples avg     0.8785    0.8698    0.8640     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 92/100 - Loss: 0.0267
Epoch 96/100 - Loss: 0.0265
Epoch 100/100 - Loss: 0.0265

===== HIERARCHICAL FUSION - CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

           0     0.9386    0.9314    0.9350      6091
           1     0.7143    0.5987    0.6514       476
           2     0.9505    0.9680    0.9592      5755
           3     0.9193    0.8896    0.9042      1268
           4     0.7979    0.7367    0.7661      1018
           5     0.9329    0.8710    0.9009      3798
           6     0.7694    0.8013    0.7850       458
           7     0.9517    0.9470    0.9494      4039

   micro avg     0.9287    0.9128    0.9207     22903
   macro avg     0.8718    0.8430    0.8564     22903
weighted avg     0.9276    0.9128    0.9198     22903
 samples avg     0.8773    0.8669    0.8622     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
